# Phase 2 — Data Cleaning & Validation

## Life Claims Intelligence

### Objective

Clean and validate the extracted Life Insurance Claims dataset while
preserving the integrity of the original raw data.

### Cleaning Workflow

1. Load and inspect the raw dataset
2. Identify missing and inconsistent values
3. Convert claim fields to appropriate numeric data types
4. Standardize insurer names
5. Check insurer-year uniqueness
6. Check for aggregate rows
7. Identify and flag low-volume observations
8. Validate negative and extreme values
9. Validate claim-count consistency
10. Remove records with no usable analytical data
11. Perform final quality checks
12. Export the cleaned dataset

### Data Integrity Principle

The raw dataset is treated as immutable.

All cleaning and transformations are performed on a separate working
DataFrame, and the cleaned dataset is saved as a separate downstream
file.

---

## 1. Load & Initial Inspection

Before cleaning the data, I will first inspect its structure, dimensions, data types, and missing values.

**Principle:** Inspect → Decide → Clean → Validate

In [1]:
import pandas as pd

In [2]:
# Raw dataset — never modify this
df_raw = pd.read_csv("D:/My Projects/Insurance_Claims_Intelligence/data/raw/Life_Claims_Raw_Extracted.csv")

# Working copy for cleaning
df_clean = df_raw.copy()

In [3]:
df_clean.head(30)

,FY,Insurer,pending_start_count,pending_start_amount,intimated_count,intimated_amount,total_claims_count,total_claims_amount,paid_count,paid_amount,...,rejected_count,rejected_amount,unclaimed_count,unclaimed_amount,pending_end_count,pending_end_amount,pending_lt_3m,pending_3_to_6m,pending_6m_to_1y,pending_gt_1y
0,2020-21,LIC,5875.0,3.496900e+02,941101.0,18755.650000,946976.0,19105.340000,933889.0,18295.580000,...,2934,3.92,1897,236.49,1725,292.4199999999984,792,933,0,0
1,2020-21,Aditya Birla Sun Life,19.0,3.802883e+00,6455.0,468.846357,6474.0,472.649240,6347.0,440.264288,...,0,0,0,0,11,3.637907523579965,10,1,0,0
2,2020-21,Aegon,0.0,0.000000e+00,401.0,107.440000,401.0,107.440000,398.0,105.980000,...,0,0,0,0,0,-6.21724893790088e-15,0,0,0,0
3,2020-21,Ageas Federal,5.0,1.255000e+00,1800.0,87.051683,1805.0,88.306683,1716.0,73.848647,...,1,0.0485005,0,0,50,6.698501542999994,50,0,0,0
4,2020-21,Aviva,5.0,7.783592e-01,1050.0,116.371251,1055.0,117.149610,1034.0,111.572118,...,0,0,0,0,0,1.06581410364015e-14,0,0,0,0
5,2020-21,Bajaj Allianz,2.0,6.500000e-01,14331.0,445.886349,14333.0,446.536349,14115.0,410.676805,...,0,0,0,0,5,3.700000007650047,5,0,0,0
6,2020-21,Bharti Axa,2.0,1.153392e+00,1891.0,106.444937,1893.0,107.598328,1875.0,106.035205,...,0,0,0,0,0,-6.483702463810914e-14,0,0,0,0
7,2020-21,Canara HSBC OBC,2.0,2.000000e+00,1897.0,166.537924,1899.0,168.537924,1844.0,156.075726,...,0,0,0,0,25,5.867974299999497,24,1,0,0
8,2020-21,Edelweiss Tokio,0.0,0.000000e+00,502.0,52.089138,502.0,52.089138,487.0,45.828280,...,0,0,0,0,2,3.000000000190504,2,0,0,0
9,2020-21,Exide Life,38.0,9.048578e+00,5014.0,173.655439,5052.0,182.704017,4978.0,170.430141,...,0,0,0,0,63,8.784217400000024,61,2,0,0


In [4]:
# Validation 

print(df_clean.shape)
print(df_clean.info())

(126, 22)
<class 'pandas.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   FY                    126 non-null    str    
 1   Insurer               126 non-null    str    
 2   pending_start_count   122 non-null    float64
 3   pending_start_amount  122 non-null    float64
 4   intimated_count       122 non-null    float64
 5   intimated_amount      122 non-null    float64
 6   total_claims_count    122 non-null    float64
 7   total_claims_amount   122 non-null    float64
 8   paid_count            122 non-null    float64
 9   paid_amount           122 non-null    float64
 10  repudiated_count      122 non-null    float64
 11  repudiated_amount     122 non-null    float64
 12  rejected_count        122 non-null    str    
 13  rejected_amount       122 non-null    str    
 14  unclaimed_count       122 non-null    str    
 15  unclaimed_amount      12

----

### Step 2 — Unusual/Missing Values Inspect

In [5]:
for col in df_clean.columns:
    print("\n",col)
    print(df_clean[col].value_counts(dropna=False).head(10))


 FY
FY
2022-23    26
2023-24    26
2024-25    26
2020-21    24
2021-22    24
Name: count, dtype: int64

 Insurer
Insurer
LIC                      5
Aditya Birla Sun Life    5
Ageas Federal            5
Aviva                    5
Bajaj Allianz            5
Bharti Axa               5
Future Generali          5
HDFC Life                5
ICICI Prudential         5
India First              5
Name: count, dtype: int64

 pending_start_count
pending_start_count
0.0     35
2.0     16
3.0      8
1.0      7
5.0      6
11.0     5
4.0      5
NaN      4
9.0      3
8.0      3
Name: count, dtype: int64

 pending_start_amount
pending_start_amount
0.000000      32
NaN            4
2.500000       2
349.690000     1
3.802883       1
1.255000       1
0.778359       1
0.650000       1
1.153392       1
2.000000       1
Name: count, dtype: int64

 intimated_count
intimated_count
0.0         6
NaN         4
941101.0    1
6455.0      1
401.0       1
1800.0      1
1050.0      1
14331.0     1
1891.0      1
1897

### Step 2A — 

In [6]:
for col in df_clean.columns:
    count = (df_clean[col].astype(str).str.strip()=="-").sum()
    if count > 0 :
        print(col,":",count)

rejected_count : 20
rejected_amount : 20
unclaimed_count : 18
unclaimed_amount : 18
pending_end_count : 4
pending_end_amount : 4
pending_lt_3m : 6
pending_3_to_6m : 14
pending_6m_to_1y : 17
pending_gt_1y : 21


In [7]:
cols_with_dash = [
    'rejected_count',
    'rejected_amount',
    'unclaimed_count',
    'unclaimed_amount',
    'pending_end_count',
    'pending_end_amount',
    'pending_lt_3m',
    'pending_3_to_6m',
    'pending_6m_to_1y',
    'pending_gt_1y'
]

for col in cols_with_dash:
    print(f"\n--- {col} ---")
    print(df_clean.loc[df_clean[col].astype(str).str.strip() == '-', ['FY', 'Insurer', col]].to_string(index=False))


--- rejected_count ---
     FY               Insurer rejected_count
2021-22 Aditya Birla Sun Life              -
2021-22                 Aegon              -
2021-22         Ageas Federal              -
2021-22                 Aviva              -
2021-22         Bajaj Allianz              -
2021-22           Bharti Axa               -
2021-22       Canara HSBC OBC              -
2021-22       Edelweiss Tokio              -
2021-22            Exide Life              -
2021-22       Future Generali              -
2021-22      ICICI Prudential              -
2021-22           India First              -
2021-22       Kotak Mahindra               -
2021-22              Max Life              -
2021-22          PNB Met Life              -
2021-22        Pramerica Life              -
2021-22       Reliance Nippon              -
2021-22             SBI Life               -
2021-22            Star Union              -
2021-22              Tata AIA              -

--- rejected_amount ---
     F

---

## 2. Handling "-" Values

The source report uses "-" in claim-related fields where no value is reported.
After cross-checking the extracted data with the original IRDAI table, these values will be treated as 0.

This allows the claim metrics to be analyzed numerically without introducing artificial missing values.

In [8]:
cols_to_clean = [
    'rejected_count',
    'rejected_amount',
    'unclaimed_count',
    'unclaimed_amount',
    'pending_end_count',
    'pending_end_amount',
    'pending_lt_3m',
    'pending_3_to_6m',
    'pending_6m_to_1y',
    'pending_gt_1y'
]

for col in cols_to_clean:
    df_clean[col] = df_clean[col].replace('-', 0)

In [9]:
print(df_clean[cols_to_clean].dtypes)

rejected_count        object
rejected_amount       object
unclaimed_count       object
unclaimed_amount      object
pending_end_count     object
pending_end_amount    object
pending_lt_3m         object
pending_3_to_6m       object
pending_6m_to_1y      object
pending_gt_1y         object
dtype: object


In [10]:
for col in cols_to_clean:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

In [11]:
print(df_clean[cols_to_clean].dtypes)

rejected_count        float64
rejected_amount       float64
unclaimed_count       float64
unclaimed_amount      float64
pending_end_count     float64
pending_end_amount    float64
pending_lt_3m         float64
pending_3_to_6m       float64
pending_6m_to_1y      float64
pending_gt_1y         float64
dtype: object


### Indian-number-format check

In [12]:
print(df_clean.dtypes)

FY                          str
Insurer                     str
pending_start_count     float64
pending_start_amount    float64
intimated_count         float64
intimated_amount        float64
total_claims_count      float64
total_claims_amount     float64
paid_count              float64
paid_amount             float64
repudiated_count        float64
repudiated_amount       float64
rejected_count          float64
rejected_amount         float64
unclaimed_count         float64
unclaimed_amount        float64
pending_end_count       float64
pending_end_amount      float64
pending_lt_3m           float64
pending_3_to_6m         float64
pending_6m_to_1y        float64
pending_gt_1y           float64
dtype: object


---

## 3. Standardizing Insurer Names

Insurer names are categorical identifiers used for grouping, trend analysis, and later merging with other Life Insurance datasets.

First, I will check for leading or trailing whitespace before making any changes.

In [13]:
for name in df_clean["Insurer"].unique():
    if name != name.strip():
        print(repr(name))

'LIC '
'Bharti Axa '
'Kotak Mahindra '
'SBI Life '


### 3.1 Removing Leading and Trailing Whitespace

Some insurer names contain unnecessary leading or trailing spaces.

These spaces can cause the same insurer to be treated as different categories during grouping, filtering, or merging.

I will remove only the unnecessary whitespace while preserving the actual insurer name.

In [14]:
df_clean["Insurer"] = df_clean["Insurer"].str.strip()

In [15]:
for name in df_clean["Insurer"].unique():
    if name != name.strip():
        print(repr(name))

In [16]:
df_clean.duplicated(
    subset=['FY', 'Insurer']
).sum()

np.int64(0)

----

## 4. Insurer Name Consistency Across Financial Years

Insurer names need to be checked across all financial years because an insurer may change its name due to rebranding, acquisition, or merger.

I will first inspect the reported names year-wise before creating any standardization or reconciliation rules.

In [17]:

for fy in sorted(df_clean["FY"].unique()):
    print(f"\n---{fy}---")
    print(sorted(df_clean.loc[df_clean["FY"]==fy,"Insurer"].unique()))


---2020-21---
['Aditya Birla Sun Life', 'Aegon', 'Ageas Federal', 'Aviva', 'Bajaj Allianz', 'Bharti Axa', 'Canara HSBC OBC', 'Edelweiss Tokio', 'Exide Life', 'Future Generali', 'HDFC Life', 'ICICI Prudential', 'India First', 'Kotak Mahindra', 'LIC', 'Max Life', 'PNB Met Life', 'Pramerica Life', 'Reliance Nippon', 'SBI Life', 'Sahara', 'Shriram', 'Star Union', 'Tata AIA']

---2021-22---
['Aditya Birla Sun Life', 'Aegon', 'Ageas Federal', 'Aviva', 'Bajaj Allianz', 'Bharti Axa', 'Canara HSBC OBC', 'Edelweiss Tokio', 'Exide Life', 'Future Generali', 'HDFC Life', 'ICICI Prudential', 'India First', 'Kotak Mahindra', 'LIC', 'Max Life', 'PNB Met Life', 'Pramerica Life', 'Reliance Nippon', 'SBI Life', 'Sahara', 'Shriram', 'Star Union', 'Tata AIA']

---2022-23---
['Acko Life', 'Aditya Birla Sun Life', 'Ageas Federal', 'Aviva', 'Bajaj Allianz', 'Bandhan', 'Bharti Axa', 'Canara HSBC', 'Credit Access Life', 'Edelweiss Tokio', 'Future Generali', 'Godigit Life', 'HDFC Life', 'ICICI Prudential', 'Ind

----

## 4.1 Year-over-Year Insurer Presence Check

I will compare insurer names across consecutive financial years to identify insurers that appear or disappear from the dataset.

This helps distinguish genuine changes in insurer coverage from possible naming inconsistencies.

In [18]:
years = sorted(df_clean['FY'].unique())

for i in range(1, len(years)):
    
    previous_year = set(
        df_clean.loc[df_clean['FY'] == years[i-1], 'Insurer']
    )
    
    current_year = set(
        df_clean.loc[df_clean['FY'] == years[i], 'Insurer']
    )
    
    disappeared = previous_year - current_year
    appeared = current_year - previous_year
    
    print(f"\n{years[i-1]} → {years[i]}")
    print("Disappeared:", sorted(disappeared))
    print("Appeared:", sorted(appeared))


2020-21 → 2021-22
Disappeared: []
Appeared: []

2021-22 → 2022-23
Disappeared: ['Aegon', 'Canara HSBC OBC', 'Exide Life']
Appeared: ['Acko Life', 'Bandhan', 'Canara HSBC', 'Credit Access Life', 'Godigit Life']

2022-23 → 2023-24
Disappeared: ['Edelweiss Tokio', 'Max Life']
Appeared: ['Axis Max Life', 'Edelweiss Life']

2023-24 → 2024-25
Disappeared: []
Appeared: []


### 4.2 Standardizing Historical Insurer Names

In [19]:
INSURER_MAP = {
    "Aegon": "Bandhan",
    "Canara HSBC OBC": "Canara HSBC",
    "Edelweiss Tokio": "Edelweiss Life",
    "Max Life": "Axis Max Life"
}

df_clean['Insurer'] = df_clean['Insurer'].replace(INSURER_MAP)

In [20]:
df_clean.duplicated(
    subset=['FY', 'Insurer']
).sum()

np.int64(0)

-----

## 5. Checking for Aggregate Rows

The source tables contain sector-level and grand-total rows in addition to individual insurers.

These aggregate rows must not be present in the insurer-level analytical dataset because they could be incorrectly treated as individual insurers and distort rankings and aggregations.

In [21]:
suspicious = df_clean[
    df_clean['Insurer'].str.contains(
        'total|sector|grand',
        case=False,
        na=False
    )
]

print(suspicious[['FY', 'Insurer']].to_string(index=False))

Empty DataFrame
Columns: [FY, Insurer]
Index: []


-----

## 6. Checking for Duplicate Insurer-Year Records

Each insurer should have one record per financial year in the extracted dataset.

I will check for duplicate combinations of Financial Year and Insurer before proceeding with further cleaning.

In [22]:
duplicates = df_clean[
    df_clean.duplicated(
        subset=['FY', 'Insurer'],
        keep=False
    )
]

print(duplicates[['FY', 'Insurer']].sort_values(['FY', 'Insurer']).to_string(index=False))

Empty DataFrame
Columns: [FY, Insurer]
Index: []


----

## 7. Identifying Low-Volume Records

Insurers with very low claim volumes may produce less stable performance rates.

Instead of removing these records, I will first inspect claim-volume distribution and create a flag for low-volume observations.

In [23]:
print(df_clean['intimated_count'].describe())

count    1.220000e+02
mean     4.759593e+04
std      1.992838e+05
min      0.000000e+00
25%      1.080000e+03
50%      3.933500e+03
75%      1.180075e+04
max      1.365379e+06
Name: intimated_count, dtype: float64


In [24]:
print(
    df_clean[['FY', 'Insurer', 'intimated_count']]
    .sort_values('intimated_count')
    .head(15)
)

          FY             Insurer  intimated_count
101  2024-25           Acko Life              0.0
86   2023-24        Godigit Life              0.0
121  2024-25              Sahara              0.0
108  2024-25  Credit Access Life              0.0
75   2023-24           Acko Life              0.0
82   2023-24  Credit Access Life              0.0
112  2024-25        Godigit Life              3.0
77   2023-24             Bandhan            295.0
51   2022-23             Bandhan            315.0
103  2024-25             Bandhan            365.0
2    2020-21             Bandhan            401.0
58   2022-23      Edelweiss Life            500.0
8    2020-21      Edelweiss Life            502.0
84   2023-24      Edelweiss Life            522.0
110  2024-25      Edelweiss Life            560.0


In [25]:
print("Zero claims:", (df_clean['intimated_count'] == 0).sum())

print("Below 100:", (df_clean['intimated_count'] < 100).sum())

print("Below 500:", (df_clean['intimated_count'] < 500).sum())

Zero claims: 6
Below 100: 7
Below 500: 11


In [26]:
print(
    df_clean.loc[
        df_clean['intimated_count'] < 500,
        ['FY', 'Insurer', 'intimated_count']
    ].sort_values(['FY', 'intimated_count'])
    .to_string(index=False)
)

     FY            Insurer  intimated_count
2020-21            Bandhan            401.0
2022-23            Bandhan            315.0
2023-24          Acko Life              0.0
2023-24 Credit Access Life              0.0
2023-24       Godigit Life              0.0
2023-24            Bandhan            295.0
2024-25          Acko Life              0.0
2024-25 Credit Access Life              0.0
2024-25             Sahara              0.0
2024-25       Godigit Life              3.0
2024-25            Bandhan            365.0


In [27]:
print(df_clean['intimated_count'].describe())

count    1.220000e+02
mean     4.759593e+04
std      1.992838e+05
min      0.000000e+00
25%      1.080000e+03
50%      3.933500e+03
75%      1.180075e+04
max      1.365379e+06
Name: intimated_count, dtype: float64


In [28]:
missing_intimated = df_clean[df_clean['intimated_count'].isna()]

print(
    missing_intimated[
        ['FY', 'Insurer', 'intimated_count']
    ].to_string(index=False)
)

     FY            Insurer  intimated_count
2022-23          Acko Life              NaN
2022-23 Credit Access Life              NaN
2022-23       Godigit Life              NaN
2023-24             Sahara              NaN


In [29]:
df_clean['Low_Volume_Flag'] = df_clean['intimated_count'] < 500
df_clean["Low_Volume_Flag"]

0      False
1      False
2       True
3      False
4      False
       ...  
121     True
122    False
123    False
124    False
125    False
Name: Low_Volume_Flag, Length: 126, dtype: bool

### Why 500 Claims?

A threshold of 500 intimated claims is used as an analytical flag rather
than a data-cleaning rule.

Low-volume insurer-year observations can produce unstable percentage
metrics because a small change in claim counts can materially change the
result.

These records are therefore retained in the dataset and flagged for
caution during downstream analysis.

----

## 8. Negative Value Validation


In [30]:
numeric_cols = df_clean.select_dtypes(include='number').columns

negative_values = {}

for col in numeric_cols:
    count = (df_clean[col] < 0).sum()
    if count > 0:
        negative_values[col] = count

negative_values

{'pending_end_amount': np.int64(13)}

In [31]:
negative_pending = df_clean[
    df_clean['pending_end_amount'] < 0
]

print(
    negative_pending[
        ['FY', 'Insurer', 'pending_end_amount']
    ].to_string(index=False)
)

     FY         Insurer  pending_end_amount
2020-21         Bandhan       -6.217249e-15
2020-21      Bharti Axa       -6.483702e-14
2022-23           Aviva       -9.325873e-15
2022-23      Bharti Axa       -3.510000e-03
2022-23  Edelweiss Life       -7.105427e-15
2022-23 Future Generali       -1.776357e-14
2022-23        Tata AIA       -3.835821e-14
2023-24           Aviva       -2.886580e-15
2023-24  Edelweiss Life       -4.226413e-03
2023-24 Future Generali       -2.992051e-14
2023-24   Axis Max Life       -3.099882e-08
2024-25 Future Generali       -1.588833e-14
2024-25   Axis Max Life       -6.850053e-08


In [32]:
negative_pending = df_clean[
    df_clean['pending_end_amount'] < 0
]

print(
    negative_pending[
        ['FY', 'Insurer', 'pending_end_count', 'pending_end_amount']
    ].to_string(index=False)
)

     FY         Insurer  pending_end_count  pending_end_amount
2020-21         Bandhan                0.0       -6.217249e-15
2020-21      Bharti Axa                0.0       -6.483702e-14
2022-23           Aviva                0.0       -9.325873e-15
2022-23      Bharti Axa                0.0       -3.510000e-03
2022-23  Edelweiss Life                0.0       -7.105427e-15
2022-23 Future Generali                0.0       -1.776357e-14
2022-23        Tata AIA                0.0       -3.835821e-14
2023-24           Aviva                0.0       -2.886580e-15
2023-24  Edelweiss Life                0.0       -4.226413e-03
2023-24 Future Generali                0.0       -2.992051e-14
2023-24   Axis Max Life                0.0       -3.099882e-08
2024-25 Future Generali                0.0       -1.588833e-14
2024-25   Axis Max Life                0.0       -6.850053e-08


In [33]:
df_clean.loc[
    df_clean['pending_end_amount'] < 0,
    'pending_end_amount'
] = 0

In [34]:
(df_clean['pending_end_amount'] < 0).sum()

np.int64(0)

----

## 9. Extreme and Impossible Value Checks


In [35]:
count_cols = [
    'pending_start_count',
    'intimated_count',
    'total_claims_count',
    'paid_count',
    'repudiated_count',
    'rejected_count',
    'unclaimed_count',
    'pending_end_count'
]

for col in count_cols:
    decimal_values = df_clean[
        df_clean[col].notna() &
        (df_clean[col] % 1 != 0)
    ]

    if not decimal_values.empty:
        print(f"\n{col}")
        print(
            decimal_values[['FY', 'Insurer', col]]
            .to_string(index=False)
        )

---

### 9.1 Claim Amount Columns

In [36]:
amount_cols = [
    'pending_start_amount',
    'intimated_amount',
    'total_claims_amount',
    'paid_amount',
    'repudiated_amount',
    'rejected_amount',
    'unclaimed_amount',
    'pending_end_amount'
]

for col in amount_cols:
    negative = df_clean[
        df_clean[col].notna() &
        (df_clean[col] < 0)
    ]

    if not negative.empty:
        print(f"\n{col}")
        print(
            negative[['FY', 'Insurer', col]]
            .to_string(index=False)
        )

### Source-verified empty records

Four insurer-year records contained no usable analytical claim data.
These records were retained during validation so that their source-level
absence could be verified, and were then removed from the cleaned dataset.

The original raw dataset remains unchanged.

In [37]:
df_clean['status_sum_count'] = (
    df_clean['paid_count']
    + df_clean['repudiated_count']
    + df_clean['rejected_count']
    + df_clean['unclaimed_count']
    + df_clean['pending_end_count']
)

df_clean['count_difference'] = (
    df_clean['total_claims_count']
    - df_clean['status_sum_count']
)

df_clean[
    ['FY', 'Insurer',
     'total_claims_count',
     'status_sum_count',
     'count_difference']
].sort_values(
    'count_difference'
).head(20)

,FY,Insurer,total_claims_count,status_sum_count,count_difference
0,2020-21,LIC,946976.0,946976.0,0.0
1,2020-21,Aditya Birla Sun Life,6474.0,6474.0,0.0
2,2020-21,Bandhan,401.0,401.0,0.0
3,2020-21,Ageas Federal,1805.0,1805.0,0.0
4,2020-21,Aviva,1055.0,1055.0,0.0
5,2020-21,Bajaj Allianz,14333.0,14333.0,0.0
6,2020-21,Bharti Axa,1893.0,1893.0,0.0
7,2020-21,Canara HSBC,1899.0,1899.0,0.0
8,2020-21,Edelweiss Life,502.0,502.0,0.0
9,2020-21,Exide Life,5052.0,5052.0,0.0


In [38]:
(df_clean['count_difference'] != 0).sum()

np.int64(4)

In [39]:
df_clean['count_difference'].describe()

count    122.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: count_difference, dtype: float64

In [40]:
mismatch = df_clean[
    df_clean['count_difference'] != 0
]

print(
    mismatch[
        [
            'FY',
            'Insurer',
            'pending_start_count',
            'intimated_count',
            'total_claims_count',
            'paid_count',
            'repudiated_count',
            'rejected_count',
            'unclaimed_count',
            'pending_end_count',
            'status_sum_count',
            'count_difference'
        ]
    ].to_string(index=False)
)

     FY            Insurer  pending_start_count  intimated_count  total_claims_count  paid_count  repudiated_count  rejected_count  unclaimed_count  pending_end_count  status_sum_count  count_difference
2022-23          Acko Life                  NaN              NaN                 NaN         NaN               NaN             NaN              NaN                NaN               NaN               NaN
2022-23 Credit Access Life                  NaN              NaN                 NaN         NaN               NaN             NaN              NaN                NaN               NaN               NaN
2022-23       Godigit Life                  NaN              NaN                 NaN         NaN               NaN             NaN              NaN                NaN               NaN               NaN
2023-24             Sahara                  NaN              NaN                 NaN         NaN               NaN             NaN              NaN                NaN               NaN    

### - 4 records with missing claim data, which were already identified and         retained  as missing during our earlier source verification.

In [41]:
valid_consistency = df_clean[
    df_clean['total_claims_count'].notna()
]

(valid_consistency['count_difference'] != 0).sum()

np.int64(0)

---

## 11. Final Missing-Value Audit

In [42]:
df_clean.isna().sum()

FY                      0
Insurer                 0
pending_start_count     4
pending_start_amount    4
intimated_count         4
intimated_amount        4
total_claims_count      4
total_claims_amount     4
paid_count              4
paid_amount             4
repudiated_count        4
repudiated_amount       4
rejected_count          4
rejected_amount         4
unclaimed_count         4
unclaimed_amount        4
pending_end_count       4
pending_end_amount      4
pending_lt_3m           4
pending_3_to_6m         4
pending_6m_to_1y        4
pending_gt_1y           4
Low_Volume_Flag         0
status_sum_count        4
count_difference        4
dtype: int64

In [44]:
empty_records = df_clean[
    df_clean.drop(columns=['FY', 'Insurer', 'Low_Volume_Flag',
                           'status_sum_count', 'count_difference'],
                 errors='ignore').isna().all(axis=1)
]

print(
    empty_records[['FY', 'Insurer']].to_string(index=False)
)

     FY            Insurer
2022-23          Acko Life
2022-23 Credit Access Life
2022-23       Godigit Life
2023-24             Sahara


In [45]:
df_clean = df_clean.drop(empty_records.index).copy()

In [46]:
df_clean.shape

(122, 25)

In [47]:
df_clean[
    df_clean['Insurer'].isin([
        'Acko Life',
        'Credit Access Life',
        'Godigit Life',
        'Sahara'
    ])
][['FY', 'Insurer']]

,FY,Insurer
19,2020-21,Sahara
43,2021-22,Sahara
69,2022-23,Sahara
75,2023-24,Acko Life
82,2023-24,Credit Access Life
86,2023-24,Godigit Life
101,2024-25,Acko Life
108,2024-25,Credit Access Life
112,2024-25,Godigit Life
121,2024-25,Sahara


### Four insurer-year records with no usable analytical claim data were removed from the cleaned dataset. The original raw dataset remains unchanged.

In [48]:
df_clean[
    (df_clean['FY'] == '2022-23') &
    (df_clean['Insurer'].isin([
        'Acko Life',
        'Credit Access Life',
        'Godigit Life'
    ]))
][['FY', 'Insurer']]

,FY,Insurer


----

## 12. Removing Temporary Validation Columns

In [49]:
df_clean = df_clean.drop(
    columns=['status_sum_count', 'count_difference']
)

In [50]:
df_clean.shape

(122, 23)

In [51]:
df_clean.isna().sum()

FY                      0
Insurer                 0
pending_start_count     0
pending_start_amount    0
intimated_count         0
intimated_amount        0
total_claims_count      0
total_claims_amount     0
paid_count              0
paid_amount             0
repudiated_count        0
repudiated_amount       0
rejected_count          0
rejected_amount         0
unclaimed_count         0
unclaimed_amount        0
pending_end_count       0
pending_end_amount      0
pending_lt_3m           0
pending_3_to_6m         0
pending_6m_to_1y        0
pending_gt_1y           0
Low_Volume_Flag         0
dtype: int64

----

## 13. Final Data Type and Structure Validation

In [52]:
# data type check 
df_clean.dtypes

FY                          str
Insurer                     str
pending_start_count     float64
pending_start_amount    float64
intimated_count         float64
intimated_amount        float64
total_claims_count      float64
total_claims_amount     float64
paid_count              float64
paid_amount             float64
repudiated_count        float64
repudiated_amount       float64
rejected_count          float64
rejected_amount         float64
unclaimed_count         float64
unclaimed_amount        float64
pending_end_count       float64
pending_end_amount      float64
pending_lt_3m           float64
pending_3_to_6m         float64
pending_6m_to_1y        float64
pending_gt_1y           float64
Low_Volume_Flag            bool
dtype: object

In [53]:
# shape 

df_clean.shape

(122, 23)

In [54]:
# duplicate key final check  

df_clean.duplicated(
    subset=['FY', 'Insurer']
).sum()

np.int64(0)

In [55]:
# FY coverage

df_clean['FY'].value_counts().sort_index()

FY
2020-21    24
2021-22    24
2022-23    23
2023-24    25
2024-25    26
Name: count, dtype: int64

----

### Converting float datatype into int 

In [56]:
count_cols = [
    'pending_start_count',
    'intimated_count',
    'total_claims_count',
    'paid_count',
    'repudiated_count',
    'rejected_count',
    'unclaimed_count',
    'pending_end_count'
]

df_clean[count_cols] = df_clean[count_cols].astype('int64')

In [57]:
df_clean.dtypes

FY                          str
Insurer                     str
pending_start_count       int64
pending_start_amount    float64
intimated_count           int64
intimated_amount        float64
total_claims_count        int64
total_claims_amount     float64
paid_count                int64
paid_amount             float64
repudiated_count          int64
repudiated_amount       float64
rejected_count            int64
rejected_amount         float64
unclaimed_count           int64
unclaimed_amount        float64
pending_end_count         int64
pending_end_amount      float64
pending_lt_3m           float64
pending_3_to_6m         float64
pending_6m_to_1y        float64
pending_gt_1y           float64
Low_Volume_Flag            bool
dtype: object

### Why are we doing this?

Simple:

Claim counts are discrete quantities, so final cleaned dataset mein unhe integer type rakhna semantically correct hai.

---

## Indian Number Format section close

In [58]:
# Amount columns are already stored as numeric values after cleaning.
# Indian-style comma formatting was removed during numeric conversion.

amount_cols = [
    'pending_start_amount',
    'intimated_amount',
    'total_claims_amount',
    'paid_amount',
    'repudiated_amount',
    'rejected_amount',
    'unclaimed_amount',
    'pending_end_amount'
]

df_clean[amount_cols].dtypes

pending_start_amount    float64
intimated_amount        float64
total_claims_amount     float64
paid_amount             float64
repudiated_amount       float64
rejected_amount         float64
unclaimed_amount        float64
pending_end_amount      float64
dtype: object

Why?

Amounts are stored as numeric values and are ready for mathematical operations.

## Cleaning Decision Log

| Issue | Decision | Reason |
|---|---|---|
| `-` placeholders | Converted to `0` where source context indicated no reported value | Required for numeric analysis |
| Insurer name whitespace | Removed using `.str.strip()` | Prevent duplicate insurer labels |
| Low claim volume | Created `Low_Volume_Flag` using `< 500` intimated claims | Low volume is an analytical consideration, not a data-quality error |
| Tiny negative `pending_end_amount` values | Converted to `0` | Values were negligible numerical residuals and corresponding `pending_end_count` was `0` |
| Completely empty insurer-year records | Removed 4 records | No usable analytical claim data was available |
| Historical insurer name changes | Standardized historical names using an explicit mapping | Enables consistent insurer-level longitudinal analysis while preserving the original raw dataset unchanged |
| Raw dataset | Kept unchanged | Maintains source-data integrity |

----

## Final QA Gate

In [59]:
print("Final Shape:", df_clean.shape)

print(
    "Duplicate FY + Insurer:",
    df_clean.duplicated(
        subset=['FY', 'Insurer']
    ).sum()
)

print(
    "Missing Values:",
    df_clean.isna().sum().sum()
)

print(
    "Negative Numeric Values:",
    (df_clean.select_dtypes(include='number') < 0).sum().sum()
)

print(
    "Financial Years:",
    sorted(df_clean['FY'].unique())
)

print(
    "Unique Insurers:",
    df_clean['Insurer'].nunique()
)

Final Shape: (122, 23)
Duplicate FY + Insurer: 0
Missing Values: 0
Negative Numeric Values: 0
Financial Years: ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
Unique Insurers: 27


----

## Save Csv

In [60]:
df_clean.to_csv(
    "Life_Claims_Cleaned.csv",
    index=False
)

In [61]:
df_check = pd.read_csv("Life_Claims_Cleaned.csv")

print("Shape:", df_check.shape)
print("Missing:", df_check.isna().sum().sum())
print("Duplicate FY + Insurer:",
      df_check.duplicated(['FY', 'Insurer']).sum())

Shape: (122, 23)
Missing: 0
Duplicate FY + Insurer: 0


----

# Thank you !